In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve


1. Загрузите файл classification.csv. В нем записаны истинные классы
объектов выборки (колонка true) и ответы некоторого классификатора (колонка predicted).

In [2]:
data = pd.read_csv('classification.csv')

2. Заполните таблицу ошибок классификации:

Для этого подсчитайте величины TP, FP, FN и TN согласно их
определениям. Например, FP — это количество объектов, имеющих
класс 0, но отнесенных алгоритмом к классу 1. Ответ в данном
вопросе — четыре числа через пробел.

In [3]:
TP = data[(data['true'] == 1) & (data['pred'] == 1)].shape[0]
TN = data[(data['true'] == 0) & (data['pred'] == 0)].shape[0]
FP = data[(data['true'] == 0) & (data['pred'] == 1)].shape[0]
FN = data[(data['true'] == 1) & (data['pred'] == 0)].shape[0]

confusion_matrix = pd.DataFrame(
    [[TP, FP],
     [FN, TN]],
    index=['Predicted Positive', 'Predicted Negative'],
    columns=['Actual Positive', 'Actual Negative']
)

print(confusion_matrix)


                    Actual Positive  Actual Negative
Predicted Positive               43               34
Predicted Negative               59               64


3. Посчитайте основные метрики качества классификатора:  
• Accuracy (доля верно угаданных) — sklearn.metrics.accuracy  
• Precision (точность) — sklearn.metrics.accuracy.precision_score  
• Recall (полнота) — sklearn.metrics.recall_score  
• F-мера — sklearn.metrics.f1_score

In [4]:
accuracy = accuracy_score(data['true'], data['pred'])
precision = precision_score(data['true'], data['pred'])
recall = recall_score(data['true'], data['pred'])
f1 = f1_score(data['true'], data['pred'])

4. Имеется четыре обученных классификатора. В файле scores.csv за-
писаны истинные классы и значения степени принадлежности по-
ложительному классу для каждого классификатора на некоторой
выборке:  
• для логистической регрессии — вероятность положительного
класса (колонка score_logreg),  
• для SVM — отступ от разделяющей поверхности (колонка score_svm),  
• для метрического алгоритма — взвешенная сумма классов со-
седей (колонка score_knn),  
• для решающего дерева — доля положительных объектов в ли-
сте (колонка score_tree).  
Загрузите этот файл.

In [5]:
data_score = pd.read_csv('scores.csv')


5. Посчитайте площадь под ROC-кривой для каждого классификатора. Какой классификатор имеет наибольшее значение метрики
AUC-ROC (укажите название столбца с ответами этого классификатора)? Воспользуйтесь функцией sklearn.metrics.roc_auc_score.

In [6]:
y_true = data_score['true']


auc_scores = {
    'score_logreg': roc_auc_score(y_true, data_score['score_logreg']),
    'score_svm': roc_auc_score(y_true, data_score['score_svm']),
    'score_knn': roc_auc_score(y_true, data_score['score_knn']),
    'score_tree': roc_auc_score(y_true, data_score['score_tree'])
}

best_model = max(auc_scores, key=auc_scores.get)
print(best_model)


score_logreg


6. Какой классификатор достигает наибольшей точности (Precision)
при полноте (Recall) не менее 70% (укажите название столбца с ответами этого классификатора)? Какое значение точности при этом
получается?

In [7]:
models = {
    'score_logreg': data_score['score_logreg'],
    'score_svm': data_score['score_svm'],
    'score_knn': data_score['score_knn'],
    'score_tree': data_score['score_tree']
}

best_model = None
best_precision = 0

for name, scores in models.items():
    precision, recall, _ = precision_recall_curve(y_true, scores)

    mask = recall >= 0.7
    if np.any(mask): 
        p = precision[mask].max()

        if p > best_precision:
            best_precision = p
            best_model = name

print(best_model, best_precision)

score_tree 0.6517857142857143
